# 🎨 Photo Restoration & Face Enhancement (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ansaftab-1/Face-restoration/blob/main/Photo_Restoration_Colab.ipynb)

This notebook runs **Face Restoration (GFPGAN)** and the **Photo Enhancer Web UI** using free Google Colab GPU acceleration (NVIDIA T4).

### ⚡ Step 1: Verify GPU Acceleration

In [ ]:
# Make sure your Colab runtime is set to GPU (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

### 📦 Step 2: Clone Repository & Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/ansaftab-1/Face-restoration.git
%cd Face-restoration

# Install system ffmpeg
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Install requirements
!pip install -q -r requirements.txt

# Install GPU-accelerated ONNX Runtime for CUDA
!pip uninstall -y onnxruntime
!pip install -q onnxruntime-gpu

### 🖼️ Step 3 (Option A): Test a Photo Directly in the Notebook
Upload an old or blurry photo. It will restore facial details and display a **Before vs. After** comparison.

In [ ]:
import os
from google.colab import files
import matplotlib.pyplot as plt
from PIL import Image

# 1. Upload photo
print("Upload your photo (.jpg, .png, .webp):")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file was uploaded.")

input_filename = list(uploaded.keys())[0]
output_filename = "enhanced_" + os.path.splitext(input_filename)[0] + ".jpg"

# 2. Run Face Enhancement using CUDA
print("\nRestoring facial features with GFPGAN...")
!python facefusion.py headless-run \
    -t "{input_filename}" \
    -o "{output_filename}" \
    --processors face_enhancer \
    --face-enhancer-model gfpgan_1.4 \
    --face-enhancer-blend 80 \
    --face-selector-mode many \
    --execution-providers cuda cpu

# 3. Display Side-by-Side Comparison
if os.path.exists(output_filename):
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(Image.open(input_filename))
    axes[0].set_title("Original Photo", fontsize=15, pad=10)
    axes[0].axis("off")

    axes[1].imshow(Image.open(output_filename))
    axes[1].set_title("Restored Photo (GFPGAN)", fontsize=15, pad=10)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    # 4. Download restored file
    files.download(output_filename)
else:
    print("Processing failed. Please check the logs above.")

### 🌐 Step 4 (Option B): Launch the Full Web UI via Public URL
Run this cell to launch the **Photo Enhancer Web UI** and get an instant public URL to access it in your browser.

In [ ]:
# Download and install Cloudflare Tunnel for a free instant public URL
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, time

# Start the Photo Enhancer FastAPI server in the background
print("Starting Photo Enhancer server...")
server_proc = subprocess.Popen(["python", "api_server.py"])
time.sleep(3)

# Start Cloudflare tunnel (click the generated trycloudflare.com URL)
print("Generating public URL...")
!cloudflared tunnel --url http://127.0.0.1:8000